# Step 3 Prototype Visualization — Google Colab

Visualizes the **9 learned prototypes** (3 per tumor class: NCR, ED, ET) from the Step 3 model.

**Before running:**
1. Add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` to Colab Secrets (🔑 icon in the left sidebar)
2. Update `JOB_NAME` in the Config cell (or leave `None` to auto-pick the latest step-3 job)
3. Set `DEMO_VOLUME_ID` to any volume id you want to inspect

**What this notebook does:**
- Downloads model weights and a data subset from S3
- Inspects prototype vectors: norms and pairwise cosine similarity
- Finds the training patch that most activates each prototype
- Shows per-prototype activation heatmaps on a chosen volume

## 0. GPU check

In [ ]:
import tensorflow as tf

print(f'TensorFlow version: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {gpus}')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

if not gpus:
    print('\n⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

## 1. Install extra dependencies

In [ ]:
%%capture
!pip install boto3 h5py scipy tqdm ipywidgets tf-keras -q

## 2. AWS credentials from Colab Secrets

In [ ]:
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = 'eu-central-1'  # change if needed

import boto3
# Quick check
boto3.client('sts').get_caller_identity()
print('AWS credentials OK')

## 3. Configuration

In [ ]:
# ── S3 ────────────────────────────────────────────────────────────────────────
S3_BUCKET        = 'your-brats2020-data'
S3_DATA_PREFIX   = 'preprocessed_data_cropped'
S3_OUTPUT_PREFIX = 'step3-outputs'
# Set to the exact job name, e.g. 'unet-proto-losses-20260430-140000'
# Leave None to auto-pick the most recent step-3 job
JOB_NAME = None

# ── Model hyperparams — must match the training run ──────────────────────────
BASE_CHANNELS    = 16
PROTOS_PER_CLASS = 3   # → 9 prototypes total
N_CLASSES        = 4
NUM_SLICES       = 128
HEIGHT, WIDTH    = 192, 160

# ── Data scanning ─────────────────────────────────────────────────────────────
# Volume ids to scan when finding closest patches.
# Keep this small (~20-30) to avoid long downloads on Colab.
SCAN_VOLUME_IDS = list(range(1, 31))

# ── Demo volume for activation maps ──────────────────────────────────────────
DEMO_VOLUME_ID = 1

# ── Local Colab paths ────────────────────────────────────────────────────────
MODEL_DIR  = '/content/step3_model'
DATA_DIR   = '/content/brats_data'
OUTPUT_DIR = '/content/prototype_vis'

for d in [MODEL_DIR, DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config OK')

## 4. Model code — write `unet3d_proto.py` to disk

In [ ]:
model_code = '''
# Use tf_keras (Keras 2 compat) so load_weights matches the TF 2.13 checkpoint format.
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers


class ConvBlock(keras.layers.Layer):
    def __init__(self, filters, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = layers.Conv3D(filters, 3, padding="same", use_bias=False,
                                   kernel_initializer="he_normal")
        self.norm1 = layers.LayerNormalization()
        self.relu1 = layers.ReLU()
        self.conv2 = layers.Conv3D(filters, 3, padding="same", use_bias=False,
                                   kernel_initializer="he_normal")
        self.norm2 = layers.LayerNormalization()
        self.relu2 = layers.ReLU()

    def call(self, x):
        x = self.relu1(self.norm1(self.conv1(x)))
        x = self.relu2(self.norm2(self.conv2(x)))
        return x


class EncoderBlock(keras.layers.Layer):
    def __init__(self, filters, **kwargs):
        super().__init__(**kwargs)
        self.conv = ConvBlock(filters)
        self.pool = layers.MaxPool3D(pool_size=(1, 2, 2))

    def call(self, x):
        skip = self.conv(x)
        return skip, self.pool(skip)


class DecoderBlock(keras.layers.Layer):
    def __init__(self, filters, **kwargs):
        super().__init__(**kwargs)
        self.upsample = layers.Conv3DTranspose(filters, kernel_size=(1, 2, 2),
                                               strides=(1, 2, 2), padding="same",
                                               kernel_initializer="he_normal")
        self.concat = layers.Concatenate()
        self.conv   = ConvBlock(filters)

    def call(self, x, skip):
        x = self.upsample(x)
        x = self.concat([x, skip])
        return self.conv(x)


class UNet3DProto(keras.Model):
    TUMOR_CLASSES = [1, 2, 3]

    def __init__(self, n_classes=4, base_channels=16, protos_per_class=3, **kwargs):
        super().__init__(**kwargs)
        c                     = base_channels
        self.protos_per_class = protos_per_class
        self.num_prototypes   = protos_per_class * len(self.TUMOR_CLASSES)
        self.proto_dim        = c * 8

        self.enc1       = EncoderBlock(c)
        self.enc2       = EncoderBlock(c * 2)
        self.enc3       = EncoderBlock(c * 4)
        self.bottleneck = ConvBlock(c * 8)
        self.dropout    = layers.Dropout(0.2)

        self.prototype_vectors = tf.Variable(
            tf.initializers.GlorotUniform()(
                shape=(self.num_prototypes, self.proto_dim, 1, 1, 1)),
            trainable=True, name="prototype_vectors"
        )
        self.prototype_to_features = layers.Conv3D(
            self.proto_dim, kernel_size=1,
            kernel_initializer="zeros", bias_initializer="zeros",
            name="prototype_to_features"
        )

        self.dec3     = DecoderBlock(c * 4)
        self.dec2     = DecoderBlock(c * 2)
        self.dec1     = DecoderBlock(c)
        self.out_conv = layers.Conv3D(n_classes, 1, kernel_initializer="glorot_uniform")

    def _l2_distances(self, x):
        proto_filters = tf.transpose(self.prototype_vectors, perm=[2, 3, 4, 1, 0])
        dot = tf.nn.conv3d(x, filters=proto_filters,
                           strides=[1, 1, 1, 1, 1], padding="SAME")
        x2  = tf.reduce_sum(tf.square(x), axis=-1, keepdims=True)
        p2  = tf.reshape(
            tf.reduce_sum(tf.square(self.prototype_vectors), axis=[1, 2, 3, 4]),
            [1, 1, 1, 1, -1])
        return tf.sqrt(tf.maximum(x2 - 2.0 * dot + p2, 1e-8))

    def _similarities(self, distances):
        return tf.math.log((distances + 1.0) / (distances + 1e-4))

    def _proto_bottleneck(self, x):
        distances    = self._l2_distances(x)
        similarities = self._similarities(distances)
        proto_feat   = self.prototype_to_features(similarities)
        return x + proto_feat, similarities, distances

    def _decode(self, x, s1, s2, s3, training):
        x = self.dropout(x, training=training)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.out_conv(x)

    def call(self, inputs, training=False):
        s1, x = self.enc1(inputs)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        x = self.bottleneck(x)
        x, _, _ = self._proto_bottleneck(x)
        return self._decode(x, s1, s2, s3, training)

    def forward_with_similarities(self, inputs):
        s1, x = self.enc1(inputs)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        x = self.bottleneck(x)
        x, similarities, _ = self._proto_bottleneck(x)
        return self._decode(x, s1, s2, s3, training=False), similarities
'''

with open('/content/unet3d_proto.py', 'w') as f:
    f.write(model_code)

import sys
sys.path.insert(0, '/content')
print('unet3d_proto.py written to /content')

## 5. Download model weights from S3

In [ ]:
import tarfile

s3 = boto3.client('s3')

if JOB_NAME is None:
    resp = s3.list_objects_v2(
        Bucket=S3_BUCKET,
        Prefix=S3_OUTPUT_PREFIX + '/',
        Delimiter='/'
    )
    jobs = sorted([
        p['Prefix'].split('/')[-2]
        for p in resp.get('CommonPrefixes', [])
        if 'unet-proto-losses' in p['Prefix']
    ])
    print('Available step-3 jobs:')
    for j in jobs:
        print(' ', j)
    if not jobs:
        raise RuntimeError('No unet-proto-losses jobs found. Set JOB_NAME manually.')
    JOB_NAME = jobs[-1]
    print(f'\nAuto-selected: {JOB_NAME}')

weights_path = os.path.join(MODEL_DIR, 'best_model.weights.h5')

if not os.path.exists(weights_path):
    # SageMaker appends /{job_name}/output/ on top of the S3OutputPath,
    # which already contained the job name — so the path has it twice.
    s3_key  = f'{S3_OUTPUT_PREFIX}/{JOB_NAME}/{JOB_NAME}/output/model.tar.gz'
    tarball = os.path.join(MODEL_DIR, 'model.tar.gz')
    print(f'Downloading s3://{S3_BUCKET}/{s3_key} …')
    s3.download_file(S3_BUCKET, s3_key, tarball)
    with tarfile.open(tarball) as t:
        t.extractall(MODEL_DIR)
    os.remove(tarball)

assert os.path.exists(weights_path)
print(f'Weights ready: {weights_path}')

## 6. Load model

In [ ]:
import numpy as np
from unet3d_proto import UNet3DProto

tf.get_logger().setLevel('ERROR')

# Use an explicit name so it matches the name used during training.
model = UNet3DProto(n_classes=N_CLASSES,
                    base_channels=BASE_CHANNELS,
                    protos_per_class=PROTOS_PER_CLASS,
                    name='u_net3d_proto')

# Build by running a dummy forward pass
_ = model(tf.zeros([1, NUM_SLICES, HEIGHT, WIDTH, 4]), training=False)

init_proto_norm = float(tf.norm(model.prototype_vectors).numpy())

# prototype_to_features Conv3D has 0 saved variables in the checkpoint (the layer
# was never built when save_weights ran).  by_name + skip_mismatch loads everything
# else (including prototype_vectors) and silently skips that one layer.
model.load_weights(weights_path, by_name=True, skip_mismatch=True)

loaded_proto_norm = float(tf.norm(model.prototype_vectors).numpy())
proto_changed = abs(loaded_proto_norm - init_proto_norm) > 1e-3

print(f'Loaded : {weights_path}')
print(f'prototype_vectors norm : {init_proto_norm:.4f} → {loaded_proto_norm:.4f}  '
      f'({"✓ weights loaded" if proto_changed else "⚠ by_name load failed — running shape-based fallback below"})')

if not proto_changed:
    # by_name failed (name mismatch between checkpoint and current session).
    # Fall back to shape-based search: find the tensor whose shape matches
    # prototype_vectors and assign it directly.
    import h5py
    target_shape = tuple(model.prototype_vectors.shape)

    def _find_and_assign(model, h5_path):
        with h5py.File(h5_path, 'r') as f:
            def search(node, path=''):
                for key in node.keys():
                    item = node[key]
                    full = f'{path}/{key}'
                    if isinstance(item, h5py.Dataset) and tuple(item.shape) == target_shape:
                        model.prototype_vectors.assign(item[:])
                        print(f'  Assigned prototype_vectors from {full}  shape={item.shape}')
                        return True
                    if isinstance(item, h5py.Group):
                        if search(item, full):
                            return True
            return search(f)

    found = _find_and_assign(model, weights_path)
    if not found:
        print('  ✗ prototype_vectors not found by shape. Run the h5-inspector cell below.')

    final_norm = float(tf.norm(model.prototype_vectors).numpy())
    print(f'  prototype_vectors norm after fallback: {init_proto_norm:.4f} → {final_norm:.4f}  '
          f'({"✓ loaded" if abs(final_norm - init_proto_norm) > 1e-3 else "⚠ still unchanged"})')

print(f'\nPrototypes : {model.num_prototypes} total  '
      f'({PROTOS_PER_CLASS} per class × 3 classes)  dim={model.proto_dim}')

In [ ]:
# ── Debug / fallback: inspect h5 structure and load prototype_vectors manually ──
# Only run this cell if the load cell above printed a ⚠ warning.

import h5py

def print_h5(node, indent=0):
    for key in node.keys():
        item = node[key]
        if isinstance(item, h5py.Group):
            print('  ' * indent + f'[{key}]')
            print_h5(item, indent + 1)
        else:
            print('  ' * indent + f'{key}: {item.shape}  dtype={item.dtype}')

with h5py.File(weights_path, 'r') as f:
    print('=== H5 file top-level structure ===')
    print_h5(f)

# --- Manual fallback loader for prototype_vectors ---
# Walks the h5 file looking for the prototype_vectors tensor and assigns it directly.
def find_and_load_prototype_vectors(model, h5_path):
    target_shape = tuple(model.prototype_vectors.shape)
    with h5py.File(h5_path, 'r') as f:
        def search(node, path=''):
            for key in node.keys():
                item = node[key]
                full = f'{path}/{key}'
                if isinstance(item, h5py.Dataset) and item.shape == target_shape:
                    data = item[:]
                    model.prototype_vectors.assign(data)
                    print(f'Assigned prototype_vectors from {full}  shape={data.shape}')
                    return True
                if isinstance(item, h5py.Group):
                    if search(item, full):
                        return True
            return False
        found = search(f)
    if not found:
        print('prototype_vectors tensor not found in h5 file — check the structure above.')

find_and_load_prototype_vectors(model, weights_path)
print(f'prototype_vectors norm after manual load: {float(tf.norm(model.prototype_vectors).numpy()):.4f}')

## 7. Prototype vector inspection

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CLASS_NAMES  = {1: 'NCR', 2: 'ED', 3: 'ET'}
CLASS_COLORS = {1: '#e07b54', 2: '#5b9bd5', 3: '#70ad47'}

pvecs      = model.prototype_vectors.numpy()          # (9, 128, 1, 1, 1)
pvecs_flat = pvecs.reshape(model.num_prototypes, -1)  # (9, 128)
norms      = np.linalg.norm(pvecs_flat, axis=1)

pvecs_norm = pvecs_flat / (norms[:, None] + 1e-8)
cosine_sim = pvecs_norm @ pvecs_norm.T

proto_labels = [
    f'{CLASS_NAMES[k // PROTOS_PER_CLASS + 1]}-{k % PROTOS_PER_CLASS}'
    for k in range(model.num_prototypes)
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bar_colors = [CLASS_COLORS[k // PROTOS_PER_CLASS + 1] for k in range(model.num_prototypes)]
axes[0].bar(proto_labels, norms, color=bar_colors)
axes[0].set_title('Prototype L2 Norms', fontsize=13)
axes[0].set_xlabel('Prototype'); axes[0].set_ylabel('L2 Norm')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)
axes[0].legend(handles=[Patch(color=CLASS_COLORS[c], label=f'{CLASS_NAMES[c]} (class {c})')
                         for c in [1, 2, 3]])

im = axes[1].imshow(cosine_sim, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_xticks(range(model.num_prototypes))
axes[1].set_xticklabels(proto_labels, rotation=45, ha='right')
axes[1].set_yticks(range(model.num_prototypes))
axes[1].set_yticklabels(proto_labels)
axes[1].set_title('Pairwise Cosine Similarity', fontsize=13)
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
for i in range(model.num_prototypes):
    for j in range(model.num_prototypes):
        axes[1].text(j, i, f'{cosine_sim[i,j]:.2f}', ha='center', va='center', fontsize=7)

plt.suptitle('Prototype Vector Properties', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'prototype_statistics.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Download data subset from S3

Downloads only the volumes needed for scanning + the demo volume.

In [ ]:
from tqdm.notebook import tqdm

needed_ids = sorted(set(SCAN_VOLUME_IDS) | {DEMO_VOLUME_ID})
total_files = len(needed_ids) * NUM_SLICES
print(f'Downloading {len(needed_ids)} volumes × {NUM_SLICES} slices = {total_files} files …')

downloaded = skipped = 0
for vol_id in tqdm(needed_ids, desc='Volumes'):
    for s in range(NUM_SLICES):
        fname = f'volume_{vol_id}_slice_{s}.h5'
        local = os.path.join(DATA_DIR, fname)
        if os.path.exists(local):
            skipped += 1
            continue
        s3.download_file(S3_BUCKET, f'{S3_DATA_PREFIX}/{fname}', local)
        downloaded += 1

print(f'Done.  Downloaded={downloaded}  Already cached={skipped}')

import h5py
from tqdm.notebook import tqdm

def load_volume(vol_id):
    img_slices, mask_slices = [], []
    for s in range(NUM_SLICES):
        with h5py.File(os.path.join(DATA_DIR, f'volume_{vol_id}_slice_{s}.h5'), 'r') as f:
            img_slices.append(f['image'][:].astype(np.float32))
            mask_slices.append(f['mask'][:].astype(np.float32))
    img  = np.stack(img_slices,  axis=0)
    mask = np.stack(mask_slices, axis=0)
    vmin, vmax = img.min(), img.max()
    if vmax - vmin > 1e-8:
        img = (img - vmin) / (vmax - vmin)
    bg   = (mask.sum(-1, keepdims=True) == 0).astype(np.float32)
    mask = np.concatenate([bg, mask], axis=-1)  # (D, H, W, 4)
    return img[None], mask[None]  # (1, D, H, W, 4)


P = model.num_prototypes

best = {k: {'sim': -np.inf, 'vol_id': None, 'depth_idx': None,
             'loc_bottleneck': None,
             'full_img_slice': None,   # full (H, W, 4) slice
             'full_mask_slice': None}  # full (H, W, 4) mask slice
        for k in range(P)}

for vol_id in tqdm(SCAN_VOLUME_IDS, desc='Scanning'):
    img, mask = load_volume(vol_id)
    _, sims   = model.forward_with_similarities(tf.constant(img))
    sims      = sims.numpy()   # (1, D, Hb, Wb, P)

    for k in range(P):
        sim_k   = sims[0, :, :, :, k]   # (D, Hb, Wb)
        max_val = sim_k.max()
        if max_val <= best[k]['sim']:
            continue
        d, hb, wb = np.unravel_index(sim_k.argmax(), sim_k.shape)
        best[k] = {
            'sim':            max_val,
            'vol_id':         vol_id,
            'depth_idx':      int(d),
            'loc_bottleneck': (int(hb), int(wb)),
            'full_img_slice': img[0, d, :, :, :].copy(),    # (H, W, 4)
            'full_mask_slice': mask[0, d, :, :, :].copy(),  # (H, W, 4)
        }

print('\nBest similarities per prototype:')
for k, info in best.items():
    cls = k // PROTOS_PER_CLASS + 1
    print(f'  {CLASS_NAMES[cls]}-proto{k % PROTOS_PER_CLASS}: '
          f'sim={info["sim"]:.3f}  vol={info["vol_id"]}  depth={info["depth_idx"]}'
          f'  loc={info["loc_bottleneck"]}')

In [ ]:
import h5py

def load_volume(vol_id):
    img_slices, mask_slices = [], []
    for s in range(NUM_SLICES):
        with h5py.File(os.path.join(DATA_DIR, f'volume_{vol_id}_slice_{s}.h5'), 'r') as f:
            img_slices.append(f['image'][:].astype(np.float32))
            mask_slices.append(f['mask'][:].astype(np.float32))
    img  = np.stack(img_slices,  axis=0)
    mask = np.stack(mask_slices, axis=0)
    vmin, vmax = img.min(), img.max()
    if vmax - vmin > 1e-8:
        img = (img - vmin) / (vmax - vmin)
    bg   = (mask.sum(-1, keepdims=True) == 0).astype(np.float32)
    mask = np.concatenate([bg, mask], axis=-1)  # (D, H, W, 4)
    return img[None], mask[None]  # (1, D, H, W, 4)


PATCH_RADIUS = 12
P = model.num_prototypes

best = {k: {'sim': -np.inf, 'vol_id': None, 'depth_idx': None,
             'loc_bottleneck': None, 'img_patch': None, 'mask_patch': None}
        for k in range(P)}

for vol_id in tqdm(SCAN_VOLUME_IDS, desc='Scanning'):
    img, mask = load_volume(vol_id)
    _, sims   = model.forward_with_similarities(tf.constant(img))
    sims      = sims.numpy()   # (1, D, Hb=24, Wb=20, 9)

    for k in range(P):
        sim_k   = sims[0, :, :, :, k]   # (D, Hb, Wb)
        max_val = sim_k.max()
        if max_val <= best[k]['sim']:
            continue
        d, hb, wb = np.unravel_index(sim_k.argmax(), sim_k.shape)
        h_img = hb * 8;  w_img = wb * 8
        h0 = max(0, h_img - PATCH_RADIUS);  h1 = min(img.shape[2], h_img + PATCH_RADIUS)
        w0 = max(0, w_img - PATCH_RADIUS);  w1 = min(img.shape[3], w_img + PATCH_RADIUS)
        best[k] = {
            'sim':           max_val,
            'vol_id':        vol_id,
            'depth_idx':     int(d),
            'loc_bottleneck':(int(hb), int(wb)),
            'img_patch':     img[0,  d, h0:h1, w0:w1, :],
            'mask_patch':    mask[0, d, h0:h1, w0:w1, :],
        }

print('\nBest similarities per prototype:')
for k, info in best.items():
    cls = k // PROTOS_PER_CLASS + 1
    print(f'  {CLASS_NAMES[cls]}-proto{k % PROTOS_PER_CLASS}: '
          f'sim={info["sim"]:.3f}  vol={info["vol_id"]}  depth={info["depth_idx"]}')

import matplotlib.patches as mpatches
MODALITIES = ['T1', 'T1ce', 'T2', 'FLAIR']

for cls in [1, 2, 3]:
    proto_ids = [k for k in range(P) if k // PROTOS_PER_CLASS + 1 == cls]
    # Rows = prototypes, cols = info + 4 modalities + T1ce+seg
    fig, axes = plt.subplots(len(proto_ids), 6, figsize=(24, 4.5 * len(proto_ids)))
    if len(proto_ids) == 1:
        axes = axes[None, :]

    for row, k in enumerate(proto_ids):
        m = best[k]
        hb, wb = m['loc_bottleneck']
        # Map bottleneck coords → image coords (×8 in H and W)
        h_img, w_img = hb * 8, wb * 8
        box_size = 16  # half-size of the marker box in image pixels

        # --- Col 0: text summary ---
        axes[row, 0].text(0.5, 0.5,
            f'Proto {k}  ({CLASS_NAMES[cls]}-{k % PROTOS_PER_CLASS})\n'
            f'Best sim : {m["sim"]:.4f}\n'
            f'Volume   : {m["vol_id"]}\n'
            f'Depth    : {m["depth_idx"]}\n'
            f'Bot. loc : {m["loc_bottleneck"]}\n'
            f'Img loc  : ({h_img}, {w_img})',
            ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='#fff8e7', alpha=0.9))
        axes[row, 0].axis('off')

        if m['full_img_slice'] is None:
            for c in range(1, 6): axes[row, c].axis('off')
            continue

        slc  = m['full_img_slice']   # (H, W, 4)  already in [0,1]
        seg  = np.argmax(m['full_mask_slice'], axis=-1).astype(float)  # (H, W)

        def add_box(ax, h_c, w_c, size, color='red'):
            """Draw a rectangle centred on (h_c, w_c) in image coords."""
            r0 = max(0, h_c - size); c0 = max(0, w_c - size)
            rh = min(slc.shape[0], h_c + size) - r0
            cw = min(slc.shape[1], w_c + size) - c0
            rect = mpatches.Rectangle(
                (c0, r0), cw, rh,
                linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)

        # --- Cols 1-4: MRI modalities (full slice + red marker box) ---
        for mod_i, mod_name in enumerate(MODALITIES):
            ch = slc[:, :, mod_i]
            # Use 1st–99th percentile for contrast stretch (handles outliers)
            lo, hi = np.percentile(ch[ch > 0], [1, 99]) if ch.max() > 0 else (0, 1)
            ch_disp = np.clip((ch - lo) / (hi - lo + 1e-8), 0, 1)
            axes[row, 1 + mod_i].imshow(ch_disp, cmap='gray', vmin=0, vmax=1)
            add_box(axes[row, 1 + mod_i], h_img, w_img, box_size)
            axes[row, 1 + mod_i].set_title(mod_name, fontsize=9)
            axes[row, 1 + mod_i].axis('off')

        # --- Col 5: T1ce + GT seg overlay ---
        ch = slc[:, :, 1]
        lo, hi = np.percentile(ch[ch > 0], [1, 99]) if ch.max() > 0 else (0, 1)
        ch_disp = np.clip((ch - lo) / (hi - lo + 1e-8), 0, 1)
        axes[row, 5].imshow(ch_disp, cmap='gray', vmin=0, vmax=1)
        axes[row, 5].imshow(np.ma.masked_where(seg == 0, seg),
                            cmap='Set1', vmin=1, vmax=3, alpha=0.5)
        add_box(axes[row, 5], h_img, w_img, box_size)
        axes[row, 5].set_title('T1ce + seg', fontsize=9)
        axes[row, 5].axis('off')

    plt.suptitle(f'Prototypes — class {cls} ({CLASS_NAMES[cls]})  '
                 f'[red box = prototype peak location]',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f'prototypes_class{cls}_{CLASS_NAMES[cls]}.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

In [ ]:
MODALITIES = ['T1', 'T1ce', 'T2', 'FLAIR']

for cls in [1, 2, 3]:
    proto_ids = [k for k in range(P) if k // PROTOS_PER_CLASS + 1 == cls]
    fig, axes = plt.subplots(len(proto_ids), 6, figsize=(22, 4 * len(proto_ids)))
    if len(proto_ids) == 1:
        axes = axes[None, :]

    for row, k in enumerate(proto_ids):
        m = best[k]
        axes[row, 0].text(0.5, 0.5,
            f'Proto {k}  ({CLASS_NAMES[cls]}-{k % PROTOS_PER_CLASS})\n'
            f'Best sim : {m["sim"]:.4f}\n'
            f'Volume   : {m["vol_id"]}\n'
            f'Depth    : {m["depth_idx"]}\n'
            f'Bot. loc : {m["loc_bottleneck"]}',
            ha='center', va='center', fontsize=9.5,
            bbox=dict(boxstyle='round', facecolor='#fff8e7', alpha=0.8))
        axes[row, 0].axis('off')

        if m['img_patch'] is None:
            for c in range(1, 6): axes[row, c].axis('off')
            continue

        patch = m['img_patch']
        for mod_i, mod_name in enumerate(MODALITIES):
            ch = patch[:, :, mod_i]
            ch_norm = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
            axes[row, 1 + mod_i].imshow(ch_norm, cmap='gray', vmin=0, vmax=1)
            axes[row, 1 + mod_i].set_title(mod_name, fontsize=9)
            axes[row, 1 + mod_i].axis('off')

        t1ce      = patch[:, :, 1]
        t1ce_norm = (t1ce - t1ce.min()) / (t1ce.max() - t1ce.min() + 1e-8)
        seg       = np.argmax(m['mask_patch'], axis=-1).astype(float)
        axes[row, 5].imshow(t1ce_norm, cmap='gray', vmin=0, vmax=1)
        axes[row, 5].imshow(np.ma.masked_where(seg == 0, seg),
                            cmap='Set1', vmin=1, vmax=3, alpha=0.55)
        axes[row, 5].set_title('T1ce + seg', fontsize=9)
        axes[row, 5].axis('off')

    plt.suptitle(f'Prototypes — class {cls} ({CLASS_NAMES[cls]})',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f'prototypes_class{cls}_{CLASS_NAMES[cls]}.png')
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## 11. Activation heatmaps on the demo volume

In [ ]:
from scipy.ndimage import zoom as nd_zoom

print(f'Loading volume {DEMO_VOLUME_ID} …')
demo_img, demo_mask = load_volume(DEMO_VOLUME_ID)

demo_logits, demo_sims = model.forward_with_similarities(tf.constant(demo_img))
demo_sims = demo_sims.numpy()[0]                              # (D, Hb, Wb, 9)
demo_pred = tf.nn.softmax(demo_logits, axis=-1).numpy()[0]   # (D, H, W, 4)
demo_seg  = np.argmax(demo_pred, axis=-1)                    # (D, H, W)
demo_img_v = demo_img[0]                                     # (D, H, W, 4)

D, Hb, Wb, _ = demo_sims.shape
demo_sims_up = nd_zoom(demo_sims, (1, HEIGHT / Hb, WIDTH / Wb, 1), order=1)  # (D, H, W, 9)

peak_depth = {k: int(demo_sims_up[:, :, :, k].max(axis=(1, 2)).argmax())
              for k in range(P)}

print('Similarities upsampled:', demo_sims.shape, '->', demo_sims_up.shape)
print('Peak depth per prototype:')
for k in range(P):
    cls = k // PROTOS_PER_CLASS + 1
    print(f'  {CLASS_NAMES[cls]}-proto{k % PROTOS_PER_CLASS}: '
          f'depth={peak_depth[k]}  '
          f'max_sim={demo_sims_up[peak_depth[k], :, :, k].max():.4f}')

In [ ]:
# Grid: rows = classes, columns = prototypes per class
fig, axes = plt.subplots(3, PROTOS_PER_CLASS, figsize=(5 * PROTOS_PER_CLASS, 14))

for cls in [1, 2, 3]:
    proto_ids = [k for k in range(P) if k // PROTOS_PER_CLASS + 1 == cls]
    for col, k in enumerate(proto_ids):
        d         = peak_depth[k]
        t1ce      = demo_img_v[d, :, :, 1]
        t1ce_norm = (t1ce - t1ce.min()) / (t1ce.max() - t1ce.min() + 1e-8)
        sim_slice = demo_sims_up[d, :, :, k]
        seg_slice = demo_seg[d]

        ax = axes[cls - 1, col]
        ax.imshow(t1ce_norm, cmap='gray', vmin=0, vmax=1)
        hm = ax.imshow(sim_slice, cmap='hot', alpha=0.45,
                       vmin=sim_slice.min(), vmax=sim_slice.max())
        ax.contour(seg_slice == cls, levels=[0.5],
                   colors=[CLASS_COLORS[cls]], linewidths=1.2)
        plt.colorbar(hm, ax=ax, fraction=0.046, pad=0.02)
        ax.set_title(f'{CLASS_NAMES[cls]}-proto{k % PROTOS_PER_CLASS}  depth={d}', fontsize=10)
        ax.axis('off')

plt.suptitle(f'Prototype Similarity Heatmaps — Volume {DEMO_VOLUME_ID}\n'
             f'(hot=high similarity  |  contour=predicted class region)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, f'activation_maps_vol{DEMO_VOLUME_ID}.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## 12. Interactive slice viewer

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

proto_options = [
    (f'Proto {k} — {CLASS_NAMES[k // PROTOS_PER_CLASS + 1]}-{k % PROTOS_PER_CLASS}', k)
    for k in range(P)
]

def show_slice(proto_idx, depth):
    t1ce      = demo_img_v[depth, :, :, 1]
    t1ce_norm = (t1ce - t1ce.min()) / (t1ce.max() - t1ce.min() + 1e-8)
    sim       = demo_sims_up[depth, :, :, proto_idx]
    seg       = demo_seg[depth]
    cls       = proto_idx // PROTOS_PER_CLASS + 1

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(t1ce_norm, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title(f'T1ce  (depth={depth})')
    axes[0].axis('off')

    axes[1].imshow(t1ce_norm, cmap='gray', vmin=0, vmax=1)
    hm = axes[1].imshow(sim, cmap='hot', alpha=0.5,
                        vmin=demo_sims_up[:, :, :, proto_idx].min(),
                        vmax=demo_sims_up[:, :, :, proto_idx].max())
    axes[1].set_title(f'{CLASS_NAMES[cls]}-proto{proto_idx % PROTOS_PER_CLASS} similarity')
    axes[1].axis('off')
    plt.colorbar(hm, ax=axes[1], fraction=0.046, pad=0.02)

    axes[2].imshow(t1ce_norm, cmap='gray', vmin=0, vmax=1)
    axes[2].imshow(np.ma.masked_where(seg == 0, seg.astype(float)),
                   cmap='Set1', vmin=1, vmax=3, alpha=0.5)
    axes[2].set_title('Predicted segmentation')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

interact(
    show_slice,
    proto_idx=Dropdown(options=proto_options, description='Prototype'),
    depth=IntSlider(min=0, max=NUM_SLICES - 1, step=1,
                    value=peak_depth[0], description='Depth')
)

## 13. Depth activation profiles

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
depths = np.arange(NUM_SLICES)

for cls in [1, 2, 3]:
    ax = axes[cls - 1]
    for k in [i for i in range(P) if i // PROTOS_PER_CLASS + 1 == cls]:
        mean_sim = demo_sims_up[:, :, :, k].mean(axis=(1, 2))
        ax.plot(depths, mean_sim,
                label=f'proto {k % PROTOS_PER_CLASS}',
                color=CLASS_COLORS[cls],
                alpha=0.5 + 0.25 * (k % PROTOS_PER_CLASS),
                linewidth=1.8)
    ax.set_ylabel('Mean similarity')
    ax.set_title(f'Class {cls} — {CLASS_NAMES[cls]}',
                 color=CLASS_COLORS[cls], fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Depth slice')
plt.suptitle(f'Prototype activation depth profile — Volume {DEMO_VOLUME_ID}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out = os.path.join(OUTPUT_DIR, f'depth_profile_vol{DEMO_VOLUME_ID}.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')